# L3c · Embeddings y reranker: comparación con tu comparación manual (máquina de desarrollo)

Compara, **con modelos reales**, la recuperación de varias configuraciones contra tu comparación hecha a mano:

| Configuración | Qué es |
|---|---|
| embeddings solos | el modelo de embeddings del stage, sin reranker |
| embeddings + `Qwen3-Reranker-0.6B` | el reranker reordena el top-N denso |
| embeddings + `bge-reranker-v2-m3` | ídem con el otro reranker |

Todas se miden igual: unión de las dos vías con los 10 mejores de cada una, **sin umbral** (así el modelo no se mezcla con el umbral), más MRR, k=1/3/5/10 y mediana de rango.
El golden set (nb 03 §6-§7) sigue siendo la validación previa; este notebook va después.

**Confidencialidad.** Tu comparación y los manuales se quedan en esta máquina (zona `val`: `documentos/val/` y `output/val/`, nunca versionadas; zona `test`: MOCK y normas públicas). Cada experimento corre en un **subproceso** (`scripts/comparar_recuperacion.py`) y guarda en `output/<ZONA>/comparaciones/` solo **ids, rangos, puntajes y métricas, nunca texto**. Antes de hacer commit: *Restart & Clear All Outputs*.

**Se puede repetir sin pagar de nuevo.** El ranking denso queda guardado por modelo (`--reusar` no vuelve a indexar) y cada puntaje del reranker queda en una caché por par: agregar un experimento o volver a correr solo calcula lo que falta. La tabla final se arma con todo lo acumulado.

**Costo.** El reranker puntúa ~20 candidatos por artículo y por sección: son cientos o miles de pares y cada uno lleva el texto completo de ambos lados. En la Raspberry tardó ~10 s por par (CPU); en el Mac usa la GPU (`mps`) si está disponible. Cada experimento imprime cuántos pares va a puntuar antes de empezar.

In [ ]:
# --- Arranque por stage (ver 00_modelos.ipynb) ---------------------------------------------------------------------------
# En el Mac: STAGE=dev en config/.env (DMR). Nada que hacer. Los rerankers usan torch/transformers (ya están en requirements).

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)

import pandas as pd
import torch
from backend.config import settings as s, carpeta_comparaciones, carpeta_manuales, carpeta_normas, carpeta_salida

print("stage:", os.environ.get("STAGE"), "| embeddings:", s.EMB_PROVIDER, s.DMR_EMB_MODEL or s.OLLAMA_EMB_MODEL or s.FOUNDRY_AI_EMBED_DEPLOYMENT)
print("dispositivo de torch: cuda" if torch.cuda.is_available() else "dispositivo de torch: mps (GPU del Mac)" if torch.backends.mps.is_available() else "dispositivo de torch: cpu (lento para el reranker)")

In [ ]:
# --- Ajusta estas constantes ---------------------------------------------------------------------------------------------
ZONA = "val"                    # "test": MOCK y normas públicas (versionable) | "val": tus documentos reales (confinados, nunca se versionan)
NORMA_ARCHIVO  = ""   # vacío = el único PDF de documentos/<ZONA>/normas/; si hay varios, lista y pide nombre o n.º
MANUAL_ARCHIVO = ""   # vacío = el único PDF de documentos/<ZONA>/manuales/; si hay varios, lista y pide nombre o n.º
ARCHIVO = carpeta_comparaciones(ZONA) / "comparacion_manual.xlsx"   # tu comparación (.csv/.xlsx) en documentos/<ZONA>/comparaciones/
COL_ARTICULO, COL_SECCION = "Artículo", "Sección"                       # nombres de columna de tu archivo
ETIQUETA = "manual_privado"      # nombre del conjunto: separa cachés y filas de la tabla (no uses el mismo para otro manual)
TOP_N = 20                       # candidatos densos que entran al reranker (>= 10)
SOLO_ARTICULOS_ETIQUETADOS = True   # True: la vía 2 compite solo con los artículos de tu archivo (como en una corrida real, donde el auditor selecciona artículos); False: con toda la norma

# Experimentos: variables de entorno del subproceso (modelo de embeddings) + reranker ("" = sin reranker). Edita, agrega o quita.
# Embeddings por Docker Model Runner. bge-m3 no está en `ai/`: se baja como GGUF de Hugging Face (Q8_0; ligeramente distinto al fp16 de la Raspberry).
# Si `docker model list` muestra otro nombre para el GGUF, copia ese nombre en DMR_EMB_MODEL.
GRANITE = {"EMB_PROVIDER": "dmr", "DMR_EMB_MODEL": "ai/granite-embedding-multilingual"}
QWEN3   = {"EMB_PROVIDER": "dmr", "DMR_EMB_MODEL": "ai/qwen3-embedding"}
BGE_M3  = {"EMB_PROVIDER": "dmr", "DMR_EMB_MODEL": "hf.co/ggml-org/bge-m3-Q8_0-GGUF"}
QWEN3_RR, BGE_RR = "Qwen/Qwen3-Reranker-0.6B", "BAAI/bge-reranker-v2-m3"
# Si copiaste los modelos a una carpeta local (sin Hugging Face), pon su ruta: QWEN3_RR = "/Users/tu/modelos/Qwen3-Reranker-0.6B"  (el nombre de la carpeta debe conservar "Qwen3-Reranker" / "bge-reranker")
EXPERIMENTOS = [
    ("granite solo",                GRANITE, ""),
    ("bge-m3 solo",                 BGE_M3,  ""),
    ("qwen3-embedding solo",        QWEN3,   ""),
    ("qwen3-embedding + qwen3-rr",  QWEN3,   QWEN3_RR),
    ("bge-m3 + bge-reranker",       BGE_M3,  BGE_RR),
    # ("granite + bge-reranker",    GRANITE, BGE_RR),
    # ("granite + qwen3-rr",        GRANITE, QWEN3_RR),
    # ("bge-m3 + qwen3-rr",         BGE_M3,  QWEN3_RR),
    # ("qwen3-embedding + bge-rr",  QWEN3,   BGE_RR),
]
# -------------------------------------------------------------------------------------------------------------------------
def resolver(nombre, carpeta, que, defecto=None):
    """PDF a usar: el nombre dado, el n.º de la lista, o (si no se da nada) el único PDF de la carpeta. Si hay varios, lista y se detiene."""
    pdfs = sorted(carpeta.glob("*.pdf"))
    if isinstance(nombre, int):
        elegido = pdfs[nombre - 1] if 1 <= nombre <= len(pdfs) else None
    elif nombre:
        elegido = carpeta / nombre if Path(nombre).name == nombre and (carpeta / nombre).is_file() else None   # solo nombres dentro de la zona
    elif len(pdfs) == 1:
        elegido = pdfs[0]
    else:
        elegido = carpeta / defecto if defecto and (carpeta / defecto).is_file() else None
    if elegido is None:
        print(f"Hay {len(pdfs)} PDF en {carpeta}. Elige la {que} (nombre o n.º) en {que.upper()}_ARCHIVO:")
        for i, f in enumerate(pdfs, 1):
            print(f"   {i}. {f.name}  ({f.stat().st_size / 2**20:.1f} MB){'  [ficticio]' if f.name.startswith('MOCK-DEMO-') else ''}")
        raise SystemExit(f"No pude elegir la {que}: {nombre or '(vacío)'}")
    return elegido
NORMA, MANUAL = resolver(NORMA_ARCHIVO, carpeta_normas(ZONA), "norma"), resolver(MANUAL_ARCHIVO, carpeta_manuales(ZONA), "manual")
assert ARCHIVO.exists(), f"Falta {ARCHIVO}: copia ahí tu comparación (carpeta de la zona val, ignorada por git)."
SALIDA = carpeta_salida(ZONA) / "comparaciones"; SALIDA.mkdir(parents=True, exist_ok=True)
carpeta_comparaciones(ZONA).mkdir(parents=True, exist_ok=True)   # aquí va tu comparación manual (CSV/XLSX)
print("Norma:", NORMA.name, "· Manual:", MANUAL.name, "· Comparación:", ARCHIVO.name, "· zona:", ZONA, "· conjunto:", ETIQUETA)

## 0 · Descarga de los modelos (una sola vez)
Descarga lo que usa la lista `EXPERIMENTOS`: los modelos de **embeddings** (Docker Model Runner; no hace falta Ollama) y los **rerankers** (Hugging Face, ~1,2 GB y ~2,2 GB; quedan en `~/.cache/huggingface`). Lo que ya está descargado no se repite. **Los rerankers no pasan por Docker Model Runner:** corren dentro del proceso con `transformers` (no hay endpoint de rerank en el DMR de dev), por eso se bajan de Hugging Face. Requisitos: Docker Desktop con *Docker Model Runner* activo (`docker model` disponible) y acceso a `huggingface.co` (si tu red lo bloquea, ver `documentos/LEEME.md`: proxy o modelos locales).

Los mismos comandos, a mano en una terminal:
```bash
docker model pull ai/granite-embedding-multilingual
docker model pull ai/qwen3-embedding
docker model pull hf.co/ggml-org/bge-m3-Q8_0-GGUF      # bge-m3 (GGUF de Hugging Face)
docker model list                                       # nombre exacto de cada modelo para DMR_EMB_MODEL
python -c "from huggingface_hub import snapshot_download as s; s('Qwen/Qwen3-Reranker-0.6B'); s('BAAI/bge-reranker-v2-m3', allow_patterns=['*.json','*.safetensors','*.model','*.txt'])"
```
**Si tu red bloquea Docker Hub (`403 Forbidden`) o `huggingface.co`:** los modelos que ya aparecen en `docker model list` se usan tal cual (la celda no los vuelve a bajar), y los rerankers se pueden copiar desde otra máquina que ya los tenga (son modelos públicos, no llevan datos). Por ejemplo, desde la Raspberry (1,2 GB + 2,2 GB):
```bash
scp -r homebridge@<IP-de-la-Pi>:'~/.cache/huggingface/hub/models--Qwen--Qwen3-Reranker-0.6B' ~/.cache/huggingface/hub/
scp -r homebridge@<IP-de-la-Pi>:'~/.cache/huggingface/hub/models--BAAI--bge-reranker-v2-m3'  ~/.cache/huggingface/hub/
```
Después ejecuta esta celda: detecta la caché y no descarga nada (si aun así transformers intenta conectarse, define `HF_HUB_OFFLINE=1` antes de abrir el notebook).


In [ ]:
import shutil

DESCARGAR = True    # False: solo muestra qué haría
embeddings_dmr    = sorted({e["DMR_EMB_MODEL"] for _, e, _ in EXPERIMENTOS if e.get("EMB_PROVIDER") == "dmr"})
embeddings_ollama = sorted({e["OLLAMA_EMB_MODEL"] for _, e, _ in EXPERIMENTOS if e.get("EMB_PROVIDER") == "ollama"})
rerankers         = sorted({rr for _, _, rr in EXPERIMENTOS if rr})

def corto(m):                                   # 'ai/qwen3-embedding:0.6B' -> 'qwen3-embedding'; 'docker model list' muestra el nombre sin 'ai/'
    return m.split("/")[-1].split(":")[0].lower()

def instalados_dmr():
    r = subprocess.run(["docker", "model", "list"], capture_output=True, text=True)
    return r.stdout.lower() if r.returncode == 0 else ""

def ejecutar(cmd):
    print("$", " ".join(cmd), flush=True)
    if DESCARGAR:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout + r.stderr).strip()[-600:])
        if r.returncode:
            print("⚠ falló. Si dice '403 Forbidden' o 'Unable to load Docker models from Docker Hub', tu red bloquea el registro: "
                  "usa los modelos que ya aparecen en `docker model list` o descarga con otra red/VPN.")

if embeddings_dmr:
    if shutil.which("docker"):
        ya = instalados_dmr()
        for m in embeddings_dmr:
            if corto(m) in ya:
                print(f"✔ {m}: ya está en `docker model list` (no se descarga). Ojo: confirma qué variante es (columna de parámetros).")
            else:
                ejecutar(["docker", "model", "pull", m])
    else:
        print("⚠ No encuentro `docker`: instala/activa Docker Desktop con Docker Model Runner.")
for m in embeddings_ollama:
    if shutil.which("ollama"):
        ejecutar(["ollama", "pull", m])
    else:
        print(f"⚠ No encuentro `ollama`: instálalo para descargar {m}.")
from huggingface_hub import snapshot_download      # ya está en el entorno (lo usa transformers)
PATRONES = ["*.json", "*.safetensors", "*.model", "*.txt"]      # solo lo necesario (sin ONNX ni otros formatos)
for m in rerankers:
    if Path(m).is_dir():
        print(f"✔ {m}: carpeta local (no se descarga)")
        continue
    try:
        print(f"✔ {m}: ya está en la caché de Hugging Face ->", snapshot_download(m, local_files_only=True, allow_patterns=PATRONES))
    except Exception:
        print(f"$ huggingface_hub.snapshot_download({m!r})", flush=True)
        if DESCARGAR:
            try:
                print("  listo:", snapshot_download(m, allow_patterns=PATRONES))
            except Exception as e:
                print(f"⚠ no pude descargar {m} ({type(e).__name__}). Si huggingface.co está bloqueado, copia la caché desde otra máquina (ver abajo).")

In [ ]:
# Corre un experimento en un subproceso (RAM aislada) y muestra su salida en vivo.
def correr(nombre, emb_env, reranker, extra=()):
    cmd = [sys.executable, "scripts/comparar_recuperacion.py", str(ARCHIVO), str(NORMA), str(MANUAL),
           "--col-articulo", COL_ARTICULO, "--col-seccion", COL_SECCION, "--zona", ZONA, "--etiqueta", ETIQUETA, "--top-n", str(TOP_N), "--reusar", *extra]
    if SOLO_ARTICULOS_ETIQUETADOS:
        cmd.append("--solo-articulos-etiquetados")
    if reranker:
        cmd += ["--reranker", reranker]
    print(f"\n{'=' * 100}\n{nombre}\n{'=' * 100}", flush=True)
    if emb_env.get("EMB_PROVIDER") == "dmr" and shutil.which("docker") and corto(emb_env["DMR_EMB_MODEL"]) not in instalados_dmr():
        print(f"⏭ Se omite: {emb_env['DMR_EMB_MODEL']} no está en `docker model list` (descárgalo en la sección 0 o quita esta fila de EXPERIMENTOS).")
        return
    env = {**os.environ, "PYTHONPATH": str(RAIZ), "PYTHONUNBUFFERED": "1", **emb_env}
    p = subprocess.Popen(cmd, cwd=RAIZ, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for linea in p.stdout:                                   # barras de progreso (\r) y log llegan tal cual
        print(linea, end="", flush=True)
    if p.wait() != 0:
        print(f"⚠ '{nombre}' terminó con error (código {p.returncode}); lo ya guardado se conserva.")

## 1 · Embeddings solos (primero, para tener el ranking denso de cada modelo)
Indexa norma y manual con cada modelo de embeddings (una vez; queda guardado) y mide el denso. Aquí aparece también el **techo del reranker**: lo máximo que puede lograr, porque solo reordena el top-N.

In [ ]:
for nombre, emb, rr in EXPERIMENTOS:
    if not rr:
        correr(nombre, emb, rr)

## 2 · Con reranker
Cada experimento imprime primero cuántos pares puntúa y cuántos ya están en la caché. **Si el total te parece mucho, interrumpe el kernel (■):** lo ya puntuado se guarda cada 50 pares y la próxima corrida sigue desde ahí.

In [ ]:
for nombre, emb, rr in EXPERIMENTOS:
    if rr:
        correr(nombre, emb, rr)

## 3 · Tabla comparada
Última medición de cada configuración, con todo lo acumulado en `output/<ZONA>/comparaciones/resultados.jsonl`. Para sumar las mediciones de otra máquina (p. ej. la Raspberry), copia sus archivos a esa carpeta y vuelve a ejecutar esta celda; las filas se separan por `conjunto`.
Lectura: con n≈75 pares, una diferencia de 1-2 puntos es ruido (mira el IC95). Decide con el **MRR** y con `k=1/3/5` (¿el par correcto sube a los primeros lugares?) y con la **unión top-10** (¿entra a los candidatos?), no con un solo número.

In [ ]:
filas = [json.loads(x) for x in (SALIDA / "resultados.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
ult = {}
for r in filas:
    ult[(r["etiqueta"], r["embeddings"], r["reranker"], r["configuracion"].split(" (")[0], r["top_n"], r.get("solo_art_etiquetados", False))] = r
t = pd.DataFrame([{
    "conjunto": r["etiqueta"], "embeddings": r["embeddings"], "reranker": r["reranker"].split("/")[-1] or "—", "configuración": r["configuracion"],
    "n": r["n"], "solo art. etiquetados": r.get("solo_art_etiquetados", False), "unión top-K %": r["union10_pct"], "K": r.get("tope", 10), "IC95": f"{r['ic95'][0]}–{r['ic95'][1]}", "v1 MRR": r["v1_mrr"], "v2 MRR": r["v2_mrr"],
    "v1 k=1/3/5/10": "/".join(str(r["v1_k"][k]) for k in ("1", "3", "5", "10")), "v2 k=1/3/5/10": "/".join(str(r["v2_k"][k]) for k in ("1", "3", "5", "10")),
    "s reranker": r.get("seg_reranker")} for r in ult.values()]).sort_values(["conjunto", "embeddings", "reranker"])
pd.set_option("display.width", 250, "display.max_columns", 20, "display.max_colwidth", 60)
t